In [1]:
import os
import time
import json
import threading
import customtkinter as ctk
from tkinter import filedialog, messagebox
import joblib
import numpy as np
from sentence_transformers import SentenceTransformer
from gtts import gTTS
from deep_translator import GoogleTranslator

# Try to import pygame for audio playback
try:
    import pygame
    pygame_available = True
except ImportError:
    pygame_available = False

ctk.set_appearance_mode("System")
ctk.set_default_color_theme("blue")

class MovieGenreModelEvaluator:
    def __init__(self):
        self.models = {
            'LogisticRegression_Optimized': joblib.load('LogisticRegression_Optimized.pkl'),
            'MLPClassifier': joblib.load('MLPClassifier.pkl')
        }
        self.mlb = joblib.load('multilabel_binarizer.pkl')
        try:
            self.thresholds = joblib.load('optimal_thresholds.pkl')
        except:
            self.thresholds = {}
        self.sbert_model = SentenceTransformer('paraphrase-mpnet-base-v2')

    def predict_with_all_models(self, summary, threshold=0.3):
        if isinstance(summary, str):
            summary_embedding = self.sbert_model.encode([summary]).reshape(1, -1)
        else:
            summary_embedding = summary

        results = {}
        for name, model in self.models.items():
            y_pred_proba = np.array([est.predict_proba(summary_embedding)[:, 1] 
                                   for est in model.estimators_]).T[0]
            genre_probs = list(zip(self.mlb.classes_, y_pred_proba))
            sorted_genre_probs = sorted(genre_probs, key=lambda x: x[1], reverse=True)
            if name == "LogisticRegression_Optimized":
                predicted_genres = [g for g, _ in sorted_genre_probs[:5]]
            else:
                predicted_genres = [g for g, p in sorted_genre_probs if p >= threshold]
            results[name] = {
                'predicted_genres': predicted_genres,
                'genre_probabilities': sorted_genre_probs
            }
        return results

class RuntimeSummaryTranslator:
    def __init__(self, output_dir=None):
        if output_dir is None:
            self.output_dir = 'translated_summaries'
        else:
            self.output_dir = output_dir
        if not os.path.exists(self.output_dir):
            os.makedirs(self.output_dir)
        self.translations_dir = os.path.join(self.output_dir, 'translations')
        self.audio_dir = os.path.join(self.output_dir, 'audio')
        for directory in [self.translations_dir, self.audio_dir]:
            if not os.path.exists(directory):
                os.makedirs(directory)
        self.language_codes = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
        self.tts_langs = {'Arabic': 'ar', 'Urdu': 'ur', 'Korean': 'ko'}
        self.translations = {}
        self.audio_paths = {}

    def translate_text(self, text, language, progress_callback=None):
        try:
            target_lang = self.language_codes[language]
            translator = GoogleTranslator(source='en', target=target_lang)
            if len(text) > 4000:
                chunks = []
                start = 0
                while start < len(text):
                    end = min(start + 3500, len(text))
                    if end < len(text):
                        boundary = max([text.rfind(x, start, end) for x in ['. ', '! ', '? ', '.\n', '!\n', '?\n']])
                        if boundary > start:
                            end = boundary + 2
                        else:
                            space = text.rfind(' ', start, end)
                            if space > start:
                                end = space + 1
                    chunks.append(text[start:end])
                    start = end
                translated_chunks = []
                for i, chunk in enumerate(chunks):
                    if progress_callback:
                        progress_callback(f"Translating chunk {i+1}/{len(chunks)} ({len(chunk)} chars)...")
                    translated_chunk = translator.translate(chunk)
                    translated_chunks.append(translated_chunk)
                    time.sleep(1.5)
                return ' '.join(translated_chunks)
            else:
                return translator.translate(text)
        except Exception as e:
            if progress_callback:
                progress_callback(f"Error translating to {language}: {e}")
            return None

    def text_to_speech(self, text, language, filename, progress_callback=None):
        try:
            lang_code = self.tts_langs[language]
            if len(text) > 10000:
                if progress_callback:
                    progress_callback(f"Audio text is very long ({len(text)} chars), breaking into chunks...")
                chunks = []
                start = 0
                while start < len(text):
                    end = min(start + 5000, len(text))
                    if end < len(text):
                        boundary = max([text.rfind(x, start, end) for x in ['. ', '! ', '? ', '.\n', '!\n', '?\n']])
                        if boundary > start:
                            end = boundary + 2
                        else:
                            space = text.rfind(' ', start, end)
                            if space > start:
                                end = space + 1
                    chunks.append(text[start:end])
                    start = end
                temp_files = []
                for i, chunk in enumerate(chunks):
                    temp_filename = f"{filename}.part{i}.mp3"
                    if progress_callback:
                        progress_callback(f"Converting audio chunk {i+1}/{len(chunks)}...")
                    tts = gTTS(text=chunk, lang=lang_code, slow=False)
                    tts.save(temp_filename)
                    temp_files.append(temp_filename)
                    time.sleep(0.5)
                import subprocess
                concat_list = "|".join(temp_files)
                try:
                    cmd = f'ffmpeg -y -i "concat:{concat_list}" -acodec copy "{filename}"'
                    subprocess.run(cmd, shell=True, check=True)
                except (subprocess.SubprocessError, FileNotFoundError):
                    with open(filename, 'wb') as outfile:
                        for temp_file in temp_files:
                            with open(temp_file, 'rb') as infile:
                                outfile.write(infile.read())
                for temp_file in temp_files:
                    try:
                        os.remove(temp_file)
                    except:
                        pass
            else:
                tts = gTTS(text=text, lang=lang_code, slow=False)
                tts.save(filename)
            return True
        except Exception as e:
            if progress_callback:
                progress_callback(f"Error creating audio for {language} ({filename}): {e}")
            return False

    def process_summary(self, summary, summary_id="summary", target_language=None, progress_callback=None):
        if progress_callback:
            progress_callback(f"Processing summary with ID: {summary_id}")
        results = {
            'summary_id': summary_id,
            'original_summary': summary,
            'translations': {},
            'audio_paths': {}
        }
        orig_summary_path = os.path.join(self.translations_dir, f"{summary_id}_original.txt")
        with open(orig_summary_path, 'w', encoding='utf-8') as f:
            f.write(summary)
        if progress_callback:
            progress_callback(f"Original summary saved to: {orig_summary_path}")
        if target_language:
            if target_language in self.language_codes:
                languages_to_process = [target_language]
            else:
                if progress_callback:
                    progress_callback(f"Warning: {target_language} is not a supported language. Processing all languages instead.")
                languages_to_process = list(self.language_codes.keys())
        else:
            languages_to_process = list(self.language_codes.keys())
        for language in languages_to_process:
            if progress_callback:
                progress_callback(f"\nTranslating to {language}...")
            translated_text = self.translate_text(summary, language, progress_callback)
            if translated_text:
                if progress_callback:
                    progress_callback(f"{language} translation complete ({len(translated_text)} characters)")
                results['translations'][language] = translated_text
                trans_file_path = os.path.join(self.translations_dir, f"{summary_id}_{language.lower()}.txt")
                with open(trans_file_path, 'w', encoding='utf-8') as f:
                    f.write(translated_text)
                if progress_callback:
                    progress_callback(f"{language} translation saved to: {trans_file_path}")
                audio_filename = f"{summary_id}_{language.lower()}.mp3"
                audio_path = os.path.join(self.audio_dir, audio_filename)
                if progress_callback:
                    progress_callback(f"Converting to {language} audio...")
                if self.text_to_speech(translated_text, language, audio_path, progress_callback):
                    results['audio_paths'][language] = audio_path
                    if progress_callback:
                        progress_callback(f"{language} audio saved to: {audio_path}")
                else:
                    if progress_callback:
                        progress_callback(f"Failed to create {language} audio")
            else:
                if progress_callback:
                    progress_callback(f"Translation to {language} failed")
            time.sleep(1.5)
        self.translations = results['translations']
        self.audio_paths = results['audio_paths']
        json_path = os.path.join(self.translations_dir, f"{summary_id}_translations.json")
        with open(json_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=4)
        if progress_callback:
            progress_callback(f"Full translation data saved to: {json_path}")
        return results

class CombinedApp(ctk.CTk):
    def __init__(self):
        super().__init__()
        self.title("Movie Genre & Translation App")
        self.geometry("950x900")
        self.minsize(700, 700)
        self.resizable(True, True)
        self.genre_evaluator = None
        self.translator = None
        self.translation_results = None
        self.audio_playing = None
        self.threshold = 0.3
        self.selected_language = ctk.StringVar(value="All")
        self.output_dir = ctk.StringVar(value="translated_summaries")
        self.summary_id = ctk.StringVar(value="movie_summary")

        # SCROLLABLE FRAME
        self.scrollable_frame = ctk.CTkScrollableFrame(self, width=920, height=880)
        self.scrollable_frame.pack(fill="both", expand=True, padx=10, pady=10)

        self._build_gui()
        threading.Thread(target=self._load_models, daemon=True).start()

    def _build_gui(self):
        ctk.CTkLabel(self.scrollable_frame, text="Enter Movie Summary:").pack(padx=20, pady=(20, 0), anchor="w")
        self.summary_text = ctk.CTkTextbox(self.scrollable_frame, width=900, height=120, font=("Arial", 13))
        self.summary_text.pack(padx=20, pady=(0, 10), fill="x")

        slider_frame = ctk.CTkFrame(self.scrollable_frame)
        slider_frame.pack(padx=20, pady=(0, 10), fill="x")
        ctk.CTkLabel(slider_frame, text="Prediction Threshold:").pack(side="left", padx=(5, 5))
        self.threshold_slider = ctk.CTkSlider(slider_frame, from_=0.0, to=1.0, number_of_steps=100, width=300, command=self._on_threshold_change)
        self.threshold_slider.set(self.threshold)
        self.threshold_slider.pack(side="left", padx=(0, 5))
        self.threshold_label = ctk.CTkLabel(slider_frame, text=f"{self.threshold:.2f}")
        self.threshold_label.pack(side="left")

        self.predict_btn = ctk.CTkButton(self.scrollable_frame, text="Predict Genres", command=self._on_predict, state="disabled", width=200)
        self.predict_btn.pack(pady=(0, 10))

        ctk.CTkLabel(self.scrollable_frame, text="Genre Output:").pack(padx=20, pady=(5, 0), anchor="w")
        self.genre_output_box = ctk.CTkTextbox(self.scrollable_frame, width=900, height=200, font=("Consolas", 12))
        self.genre_output_box.pack(padx=20, pady=(0, 10), fill="both", expand=False)
        self.genre_output_box.configure(state="disabled")

        ctk.CTkLabel(self.scrollable_frame, text="Translation & Audio", font=("Arial", 15, "bold")).pack(padx=20, pady=(10, 0), anchor="w")

        dir_frame = ctk.CTkFrame(self.scrollable_frame)
        dir_frame.pack(padx=20, pady=(10, 5), fill="x")
        ctk.CTkLabel(dir_frame, text="Output Directory:").pack(side="left", padx=(5, 5))
        dir_entry = ctk.CTkEntry(dir_frame, textvariable=self.output_dir, width=300)
        dir_entry.pack(side="left", padx=(0, 5))
        ctk.CTkButton(dir_frame, text="Browse", command=self.browse_dir).pack(side="left")

        id_frame = ctk.CTkFrame(self.scrollable_frame)
        id_frame.pack(padx=20, pady=5, fill="x")
        ctk.CTkLabel(id_frame, text="Summary Identifier:").pack(side="left", padx=(5, 5))
        ctk.CTkEntry(id_frame, textvariable=self.summary_id, width=200).pack(side="left")

        lang_frame = ctk.CTkFrame(self.scrollable_frame)
        lang_frame.pack(padx=20, pady=5, fill="x")
        ctk.CTkLabel(lang_frame, text="Target Language:").pack(side="left", padx=(5, 5))
        self.lang_options = ["All", "Arabic", "Urdu", "Korean"]
        ctk.CTkOptionMenu(lang_frame, variable=self.selected_language, values=self.lang_options).pack(side="left")

        self.translate_btn = ctk.CTkButton(self.scrollable_frame, text="Translate & Convert to Audio", command=self._on_translate, state="disabled", width=300)
        self.translate_btn.pack(pady=(0, 10))

        ctk.CTkLabel(self.scrollable_frame, text="Translation Progress:").pack(padx=20, pady=(5, 0), anchor="w")
        self.progress_box = ctk.CTkTextbox(self.scrollable_frame, width=900, height=60, font=("Arial", 11))
        self.progress_box.pack(padx=20, pady=(0, 10), fill="both", expand=False)
        self.progress_box.configure(state="disabled")

        ctk.CTkLabel(self.scrollable_frame, text="Translation Results:").pack(padx=20, pady=(5, 0), anchor="w")
        self.results_box = ctk.CTkTextbox(self.scrollable_frame, width=900, height=180, font=("Arial", 12))
        self.results_box.pack(padx=20, pady=(0, 10), fill="both", expand=False)
        self.results_box.configure(state="disabled")

        self.audio_frame = ctk.CTkFrame(self.scrollable_frame)
        self.audio_frame.pack(padx=20, pady=(5, 10), fill="x")
        self.audio_buttons = {}
        for lang in ["Arabic", "Urdu", "Korean"]:
            btn = ctk.CTkButton(self.audio_frame, text=f"Play {lang} Audio", command=lambda l=lang: self.play_audio(l), state="disabled")
            btn.pack(side="left", padx=10)
            self.audio_buttons[lang] = btn
        self.stop_btn = ctk.CTkButton(self.audio_frame, text="Stop Audio", command=self.stop_audio, state="disabled")
        self.stop_btn.pack(side="left", padx=10)

    def _load_models(self):
        self._set_genre_output("Loading models and components, please wait...\n")
        try:
            self.genre_evaluator = MovieGenreModelEvaluator()
            self.translator = RuntimeSummaryTranslator()
            self._set_genre_output("All models loaded successfully!\n")
            self.predict_btn.configure(state="normal")
            self.translate_btn.configure(state="normal")
        except Exception as e:
            self._set_genre_output(f"Error loading models: {e}\n")
            messagebox.showerror("Error", f"Error loading models: {e}")

    def _on_threshold_change(self, value):
        self.threshold = float(value)
        self.threshold_label.configure(text=f"{self.threshold:.2f}")

    def _on_predict(self):
        summary = self.summary_text.get("1.0", "end").strip()
        if not summary:
            messagebox.showerror("Error", "Please enter a movie summary.")
            return
        self.predict_btn.configure(state="disabled")
        self._set_genre_output("Predicting genres, please wait...\n")
        threading.Thread(target=self._predict_thread, args=(summary,), daemon=True).start()

    def _predict_thread(self, summary):
        try:
            results = self.genre_evaluator.predict_with_all_models(summary, threshold=self.threshold)
            output = ""
            for name, res in results.items():
                output += f"\n{name} Predictions:\n"
                output += f"Predicted Genres: {', '.join(res['predicted_genres'])}\n"
                output += "Top 5 Genre Probabilities:\n"
                for genre, prob in res['genre_probabilities'][:5]:
                    output += f"  - {genre}: {prob:.4f}\n"
            self._set_genre_output(output)
        except Exception as e:
            self._set_genre_output(f"Error during prediction: {e}\n")
            messagebox.showerror("Error", f"Error during prediction: {e}")
        finally:
            self.predict_btn.configure(state="normal")

    def _set_genre_output(self, text):
        self.genre_output_box.configure(state="normal")
        self.genre_output_box.delete("1.0", "end")
        self.genre_output_box.insert("end", text)
        self.genre_output_box.configure(state="disabled")

    def browse_dir(self):
        dir_selected = filedialog.askdirectory()
        if dir_selected:
            self.output_dir.set(dir_selected)

    def _on_translate(self):
        summary = self.summary_text.get("1.0", "end").strip()
        if not summary:
            messagebox.showerror("Error", "Please enter a movie summary.")
            return
        for btn in self.audio_buttons.values():
            btn.configure(state="disabled")
        self.stop_btn.configure(state="disabled")
        self.progress_box.configure(state="normal")
        self.progress_box.delete("1.0", "end")
        self.progress_box.insert("end", "Starting translation...\n")
        self.progress_box.configure(state="disabled")
        self.results_box.configure(state="normal")
        self.results_box.delete("1.0", "end")
        self.results_box.configure(state="disabled")
        self.translator = RuntimeSummaryTranslator(output_dir=self.output_dir.get())
        summary_id = self.summary_id.get().strip() or "movie_summary"
        lang = self.selected_language.get()
        target_language = None if lang == "All" else lang
        threading.Thread(target=self._run_translation, args=(summary, summary_id, target_language), daemon=True).start()

    def _run_translation(self, summary, summary_id, target_language):
        def progress_callback(msg):
            self.progress_box.configure(state="normal")
            self.progress_box.insert("end", msg + "\n")
            self.progress_box.see("end")
            self.progress_box.configure(state="disabled")
        results = self.translator.process_summary(summary, summary_id, target_language, progress_callback)
        self.translation_results = results
        self._display_translation_results()

    def _display_translation_results(self):
        self.results_box.configure(state="normal")
        self.results_box.delete("1.0", "end")
        if not self.translation_results:
            self.results_box.insert("end", "No translations available.")
        else:
            for language, translation in self.translation_results['translations'].items():
                self.results_box.insert("end", f"{language} Translation:\n{'-'*40}\n{translation}\n\n")
                if language in self.translation_results['audio_paths']:
                    self.results_box.insert("end", f"{language} Audio File: {self.translation_results['audio_paths'][language]}\n\n")
                self.audio_buttons[language].configure(state="normal")
            self.stop_btn.configure(state="normal")
        self.results_box.configure(state="disabled")

    def play_audio(self, language):
        if not self.translation_results or language not in self.translation_results['audio_paths']:
            messagebox.showinfo("Audio Not Available", f"No {language} audio available.")
            return
        audio_path = self.translation_results['audio_paths'][language]
        if not pygame_available:
            messagebox.showerror("Missing Dependency", "pygame is required for in-app audio playback. Please install it with:\n\npip install pygame")
            return
        try:
            if self.audio_playing:
                self.stop_audio()
            pygame.mixer.init()
            pygame.mixer.music.load(audio_path)
            pygame.mixer.music.play()
            self.audio_playing = language
        except Exception as e:
            messagebox.showerror("Error", f"Could not play audio: {e}")

    def stop_audio(self):
        if pygame_available and self.audio_playing:
            try:
                pygame.mixer.music.stop()
                self.audio_playing = None
            except Exception:
                pass

if __name__ == "__main__":
    app = CombinedApp()
    app.mainloop()

pygame 2.6.1 (SDL 2.28.4, Python 3.13.3)
Hello from the pygame community. https://www.pygame.org/contribute.html
